# QC Summary

This notebook performs quality control checks on the cardiovascular association table and the gnomAD frequency table, covering source metadata, coordinate and genome build handling, variant identifier availability and concordance, statistical field completeness, duplication, and trait classification.

In [1]:
from pathlib import Path
import numpy as np
import openpyxl
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
associations_path = Path("../data/final/association_results.csv")
gnomad_path       = Path("../data/final/gnomad_frequencies.csv")

out_qc_xlsx = Path("../data/final/qc_summary.xlsx")
out_qc_xlsx.parent.mkdir(parents=True, exist_ok=True)

In [3]:
associations_df = pd.read_csv(associations_path, low_memory=False, sep=";")
gnomad_df       = pd.read_csv(gnomad_path,       low_memory=False, sep=";")

In [4]:
dataset_metadata = pd.DataFrame([
    {
        "source_dataset": "GWAS Catalog",
        "genome_build": "No coordinates reported",
        "coordinates_original_or_lifted": "Not available",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Not applicable",
    },
    {
        "source_dataset": "HERMES",
        "genome_build": "GRCh37",
        "coordinates_original_or_lifted": "Original",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Main table excludes UK Biobank samples",
    },
    {
        "source_dataset": "CARDIoGRAMplusC4D",
        "genome_build": "GRCh38",
        "coordinates_original_or_lifted": "Harmonized",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Not applicable",
    },
    {
        "source_dataset": "FinnGen",
        "genome_build": "GRCh38",
        "coordinates_original_or_lifted": "Original",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Not applicable",
    },
])

gnomad_metadata = pd.DataFrame([
    {
        "source_dataset": "gnomAD",
        "genome_build": "GRCh38",
        "coordinates_original_or_lifted": "Original",
        "both_GRCh37_and_GRCh38_available": "No",
        "sample_overlap_flag": "Not applicable",
    },
])

In [5]:
qc_df = associations_df.copy()

qc_df = qc_df.merge(
    dataset_metadata,
    on="source_dataset",
    how="left"
)

qc_df["p_value"]          = pd.to_numeric(qc_df["p_value"],          errors="coerce")
qc_df["effect_size"]      = pd.to_numeric(qc_df["effect_size"],      errors="coerce")
qc_df["standard_error"]   = pd.to_numeric(qc_df["standard_error"],   errors="coerce")
qc_df["odds_ratio"]       = pd.to_numeric(qc_df["odds_ratio"],       errors="coerce")
qc_df["allele_frequency"]  = pd.to_numeric(qc_df["allele_frequency"], errors="coerce")

qc_gnomad = gnomad_df.copy()

if "source_dataset" not in qc_gnomad.columns:
    qc_gnomad["source_dataset"] = "gnomAD"

qc_gnomad["MAX_freq"] = pd.to_numeric(qc_gnomad["MAX_freq"], errors="coerce")

In [6]:
total_rows        = len(qc_df)
total_gnomad_rows = len(qc_gnomad)

def missing_result(df, column):
    total   = len(df)
    if column not in df.columns:
        return "column not available"
    missing = int(df[column].isna().sum())
    present = int(df[column].notna().sum())
    return f"{present}/{total} rows available; {missing}/{total} rows missing"

def metadata_summary(meta_df, column):
    return "; ".join(
        meta_df.apply(
            lambda row: f"{row['source_dataset']}: {row[column]}",
            axis=1
        )
    )

In [7]:
# Concordance: rsID-position within same genome build
position_check_df = qc_df.dropna(
    subset=["rsID", "region_assembly", "chromosome", "position"]
).copy()

position_check_df["variant_position_key"] = (
    position_check_df["chromosome"].astype(str)
    + ":"
    + position_check_df["position"].astype(str)
)

position_concordance = (
    position_check_df
    .groupby(["rsID", "region_assembly"])["variant_position_key"]
    .nunique()
    .reset_index(name="unique_positions_same_build")
)

position_conflicts = position_concordance[
    position_concordance["unique_positions_same_build"] > 1
]

# Concordance: rsID-allele pair within same genome build
allele_check_df = qc_df.dropna(
    subset=["rsID", "region_assembly", "effect_allele", "other_allele"]
).copy()

allele_check_df["allele_pair"] = allele_check_df.apply(
    lambda row: "/".join(sorted([
        str(row["effect_allele"]).upper(),
        str(row["other_allele"]).upper()
    ])),
    axis=1
)

allele_concordance = (
    allele_check_df
    .groupby(["rsID", "region_assembly"])["allele_pair"]
    .nunique()
    .reset_index(name="unique_allele_pairs_same_build")
)

allele_conflicts = allele_concordance[
    allele_concordance["unique_allele_pairs_same_build"] > 1
]

In [8]:
# Zero p-values, duplication, GWAS Catalog classification
zero_pvalue_rows = qc_df[qc_df["p_value"] == 0].copy()

zero_pvalue_by_source = (
    zero_pvalue_rows["source_dataset"]
    .value_counts()
    .reset_index()
)
zero_pvalue_by_source.columns = ["source_dataset", "count"]

fully_duplicated_rows = int(qc_df[associations_df.columns].duplicated().sum())

repeated_rsid_summary = (
    qc_df
    .dropna(subset=["rsID"])
    .groupby("rsID")
    .agg(
        total_rows=("rsID", "size"),
        source_datasets=("source_dataset", lambda x: "; ".join(sorted(set(x.dropna())))),
        traits=("trait_name", lambda x: "; ".join(sorted(set(x.dropna())))),
    )
    .reset_index()
)
repeated_rsid_summary = repeated_rsid_summary[
    repeated_rsid_summary["total_rows"] > 1
].sort_values("total_rows", ascending=False)

selected_trait_terms = [
    "heart failure",
    "coronary artery disease",
    "myocardial infarction",
]

gwas_df = qc_df[qc_df["source_dataset"] == "GWAS Catalog"].copy()

gwas_df["is_selected_trait"] = gwas_df["trait_name"].apply(
    lambda trait_name: (
        pd.notna(trait_name)
        and any(term in str(trait_name).lower() for term in selected_trait_terms)
    )
)

direct_cardiovascular_trait_rows = int(gwas_df["is_selected_trait"].sum())
broader_trait_rows               = int((~gwas_df["is_selected_trait"]).sum())

In [9]:
# Odds ratio availability and derivability
qc_df["calculated_odds_ratio"] = np.exp(qc_df["effect_size"])

qc_df["or_absolute_difference"] = (
    qc_df["odds_ratio"] - qc_df["calculated_odds_ratio"]
).abs()

rows_with_effect_size                        = int(qc_df["effect_size"].notna().sum())
rows_with_odds_ratio                         = int(qc_df["odds_ratio"].notna().sum())
rows_with_effect_size_but_missing_odds_ratio  = int(
    (qc_df["effect_size"].notna() & qc_df["odds_ratio"].isna()).sum()
)

max_or_difference = qc_df["or_absolute_difference"].max(skipna=True)

In [10]:
# gnomAD-specific computations

gnomad_unique_rsids = int(qc_gnomad["rsID"].nunique())          if "rsID"         in qc_gnomad.columns else None
gnomad_unique_genes = int(qc_gnomad["gene_symbol"].nunique())   if "gene_symbol"  in qc_gnomad.columns else None

gnomad_freq_source_breakdown = (
    qc_gnomad["MAX_freq_source"].value_counts(dropna=False)
    if "MAX_freq_source" in qc_gnomad.columns
    else pd.Series(dtype=int)
)

gnomad_priority_breakdown = (
    qc_gnomad["Priority"].value_counts(dropna=False)
    if "Priority" in qc_gnomad.columns
    else pd.Series(dtype=int)
)

n_gnomad_zero_freq    = int((qc_gnomad["MAX_freq"] == 0).sum())  if "MAX_freq" in qc_gnomad.columns else None
n_gnomad_missing_freq = int(qc_gnomad["MAX_freq"].isna().sum())  if "MAX_freq" in qc_gnomad.columns else None

if "variant_id" in qc_gnomad.columns:
    vid_series       = qc_gnomad["variant_id"].dropna()
    n_colon_fmt      = int(vid_series.str.contains(":",  na=False).sum())
    n_underscore_fmt = int(vid_series.str.contains("-",  na=False).sum())
    n_vid_missing    = int(qc_gnomad["variant_id"].isna().sum())
else:
    n_colon_fmt = n_underscore_fmt = n_vid_missing = None

In [11]:
all_metadata = pd.concat([dataset_metadata, gnomad_metadata], ignore_index=True)

qc_rows = [
    # GWAS datasets and gnomAD
    {
        "source": "GWAS datasets and gnomAD",
        "check": "Source dataset recorded for each row",
        "result": (
            f"All {total_rows} association rows and all {total_gnomad_rows} gnomAD rows have source_dataset recorded"
            if qc_df["source_dataset"].notna().all() and qc_gnomad["source_dataset"].notna().all()
            else (
                f"{int(qc_df['source_dataset'].isna().sum())}/{total_rows} association rows missing source_dataset; "
                f"{int(qc_gnomad['source_dataset'].isna().sum())}/{total_gnomad_rows} gnomAD rows missing source_dataset"
            )
        ),
    },
    {
        "source": "GWAS datasets and gnomAD",
        "check": "Genome build used by each dataset recorded",
        "result": metadata_summary(all_metadata, "genome_build"),
    },
    {
        "source": "GWAS datasets and gnomAD",
        "check": "Coordinates recorded as original or lifted over",
        "result": metadata_summary(all_metadata, "coordinates_original_or_lifted"),
    },
    {
        "source": "GWAS datasets and gnomAD",
        "check": "Availability of both GRCh37 and GRCh38 coordinates recorded",
        "result": metadata_summary(all_metadata, "both_GRCh37_and_GRCh38_available"),
    },
    {
        "source": "GWAS datasets and gnomAD",
        "check": "Potential sample overlap clearly flagged",
        "result": metadata_summary(all_metadata, "sample_overlap_flag"),
    },

    # GWAS datasets
    {
        "source": "GWAS datasets",
        "check": "rsIDs, alleles, and positions checked for availability and concordance",
        "result": (
            f"{int(qc_df['rsID'].isna().sum())} rows missing rsID; "
            f"{int(qc_df['chromosome'].isna().sum())} rows missing chromosome; "
            f"{int(qc_df['position'].isna().sum())} rows missing position; "
            f"{int(qc_df['effect_allele'].isna().sum())} rows missing effect_allele; "
            f"{int(qc_df['other_allele'].isna().sum())} rows missing other_allele; "
            f"{len(position_conflicts)} rsID-position conflicts within same build; "
            f"{len(allele_conflicts)} rsID-allele-pair conflicts within same build"
        ),
    },
    {
        "source": "GWAS datasets",
        "check": "Effect allele and non-effect allele clearly defined",
        "result": (
            f"effect_allele: {int(qc_df['effect_allele'].notna().sum())}/{total_rows} available, "
            f"{int(qc_df['effect_allele'].isna().sum())} missing; "
            f"other_allele: {int(qc_df['other_allele'].notna().sum())}/{total_rows} available, "
            f"{int(qc_df['other_allele'].isna().sum())} missing"
        ),
    },
    {
        "source": "GWAS datasets",
        "check": "p-values, effect sizes, standard errors, and allele frequencies available",
        "result": (
            f"p_value: {missing_result(qc_df, 'p_value')}; "
            f"effect_size: {missing_result(qc_df, 'effect_size')}; "
            f"standard_error: {missing_result(qc_df, 'standard_error')}; "
            f"allele_frequency: {missing_result(qc_df, 'allele_frequency')}"
        ),
    },
    {
        "source": "GWAS datasets",
        "check": "p-values reported as zero checked",
        "result": (
            f"{len(zero_pvalue_rows)} rows with p_value = 0. "
            "Breakdown by source: "
            + "; ".join(
                f"{r['source_dataset']}: {r['count']} rows"
                for _, r in zero_pvalue_by_source.iterrows()
            )
            if len(zero_pvalue_rows) > 0
            else "No rows with p_value = 0"
        ),
    },
    {
        "source": "GWAS datasets",
        "check": "Same variant appearing multiple times across traits or datasets checked",
        "result": f"{len(repeated_rsid_summary)} rsIDs appear in more than one association row",
    },
    {
        "source": "GWAS datasets",
        "check": "Fully duplicated rows checked",
        "result": f"{fully_duplicated_rows} fully duplicated rows",
    },
    {
        "source": "GWAS datasets",
        "check": "GWAS Catalog associations checked for direct cardiovascular traits or broader molecular/biomarker traits",
        "result": (
            f"{direct_cardiovascular_trait_rows} direct cardiovascular GWAS Catalog rows; "
            f"{broader_trait_rows} broader or non-selected GWAS Catalog rows"
        ),
    },
    {
        "source": "GWAS datasets",
        "check": "Odds ratios available or derivable from effect size",
        "result": (
            f"{rows_with_effect_size} rows with effect_size; "
            f"{rows_with_odds_ratio} rows with odds_ratio; "
            f"{rows_with_effect_size_but_missing_odds_ratio} rows with effect_size but missing odds_ratio; "
            f"maximum OR difference from exp(effect_size): {max_or_difference:.6f}"
            if max_or_difference is not None and not np.isnan(max_or_difference)
            else (
                f"{rows_with_effect_size} rows with effect_size; "
                f"{rows_with_odds_ratio} rows with odds_ratio; "
                f"{rows_with_effect_size_but_missing_odds_ratio} rows with effect_size but missing odds_ratio; "
                "maximum OR difference from exp(effect_size): not calculable"
            )
        ),
    },

    # gnomAD
    {
        "source": "gnomAD",
        "check": "Table row count, unique rsIDs, and unique genes",
        "result": (
            f"Total rows: {total_gnomad_rows}"
            + (f"; unique rsIDs: {gnomad_unique_rsids}" if gnomad_unique_rsids is not None else "")
            + (f"; unique genes: {gnomad_unique_genes}" if gnomad_unique_genes is not None else "")
        ),
    },
    {
        "source": "gnomAD",
        "check": "Maximum allele frequency field completeness",
        "result": missing_result(qc_gnomad, "MAX_freq"),
    },
    {
        "source": "gnomAD",
        "check": "Frequency source: genome sequencing vs exome sequencing",
        "result": (
            "; ".join(
                f"{src if str(src) != 'nan' else 'Not available (no gnomAD record found)'}: {n} rows"
                for src, n in gnomad_freq_source_breakdown.items()
            )
            if not gnomad_freq_source_breakdown.empty
            else "MAX_freq_source column not present"
        ),
    },
    {
        "source": "gnomAD",
        "check": "Priority tier classification",
        "result": (
            "; ".join(f"{tier}: {n} rows" for tier, n in gnomad_priority_breakdown.items())
            + ". Variants classified as Below threshold have a gnomAD frequency below "
            "the analysis thresholds but are still present in gnomAD with a recorded frequency."
            if not gnomad_priority_breakdown.empty
            else "Priority column not present"
        ),
    },
    {
        "source": "gnomAD",
        "check": "Variants with frequency equal to zero checked",
        "result": (
            f"{n_gnomad_zero_freq} rows with MAX_freq = 0 "
            "(variant present in gnomAD but observed at zero frequency at reported sequencing depth, "
            "distinct from variants absent from gnomAD); "
            f"{n_gnomad_missing_freq} rows with missing MAX_freq (no gnomAD record found)"
            if n_gnomad_zero_freq is not None
            else "MAX_freq column not present"
        ),
    },
    {
        "source": "gnomAD",
        "check": "Functional consequence availability",
        "result": missing_result(qc_gnomad, "Functional_consequence"),
    },
    {
        "source": "gnomAD",
        "check": "Variant ID format consistency",
        "result": (
            f"{n_colon_fmt} rows use colon-separated format (chr:pos:ref:alt); "
            f"{n_underscore_fmt} rows use dash-separated format (chr-pos-ref-alt); "
            f"{n_vid_missing} rows missing variant_id"
            if n_colon_fmt is not None
            else "variant_id column not present"
        ),
    },
]

qc_summary_df = pd.DataFrame(qc_rows)

In [12]:
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "QC Summary"

ws.append(list(qc_summary_df.columns))

for row in qc_summary_df.itertuples(index=False):
    ws.append(list(row))

wb.save(out_qc_xlsx)